# Lección 6.2 — Bagging y Random Forest: un bosque de árboles distintos

> **Módulo 6 · Ensambles: la sabiduría de la multitud** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 6.1 (por qué combinar modelos), Lección 5.4 (árboles de decisión), Lección 4.6 (validación cruzada)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es una **muestra bootstrap** y por qué deja fuera a ~1 de cada 3 ejemplos.
- Describir cómo **bagging** convierte muchos árboles inestables en un modelo estable.
- Explicar el truco extra de **Random Forest** (`max_features`) y por qué árboles *peores* pueden formar un bosque *mejor*.
- Usar el **error OOB** como validación "gratis".
- Predecir el efecto de `n_estimators`, `max_features`, `max_depth`/`min_samples_leaf` y `bootstrap`.
- Desconfiar de la importancia de variables por impureza y usar la **importancia por permutación**.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.datasets import make_moons

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: cien residentes, cien archivos distintos

Un hospital quiere un diagnóstico confiable. Tiene un archivo con 1,000 historias clínicas y **100 médicos residentes**. Si todos estudian **el mismo** archivo, aprenderán lo mismo y se equivocarán igual (los "clones" de la Lección 6.1).

Al director se le ocurre algo ingenioso: a cada residente le arma su **propio archivo** sacando historias **al azar, con reemplazo**: saca una historia, la fotocopia, la **devuelve** al cajón y vuelve a sacar, hasta juntar 1,000. Así:
- cada residente ve un archivo **del mismo tamaño** que el original;
- algunas historias le salen **repetidas** y otras **no le tocan**;
- cada residente aprende algo **un poco distinto**.

Ante un paciente nuevo, los 100 votan. Esto es **bagging**: *Bootstrap AGGregatING* (muestrear con reemplazo + combinar).

### 🔮 Ejercicio 1 — ¿Cuántos se repiten?
En un grupo hay 10 estudiantes. Sacamos 10 nombres de una urna **devolviendo** cada papelito después de sacarlo.

¿Cuántos nombres **distintos** esperas ver, en promedio?
- a) 10, todos · b) unos 6–7 · c) unos 3

✍️ *Mi apuesta:*

In [ ]:
nombres = ["Ana", "Beto", "Carla", "Dani", "Eva", "Fede", "Gaby", "Hugo", "Iris", "Juan"]
rng = np.random.default_rng(2)

muestra = rng.choice(nombres, size=len(nombres), replace=True)    # muestra bootstrap
print("Muestra bootstrap:", [str(nombre) for nombre in muestra])
print("Distintos:", len(set(muestra)), "· Se quedaron fuera:", sorted(set(nombres) - set(muestra)))

Una sola muestra depende de la suerte. Repitamos el experimento muchas veces y con grupos de distintos tamaños:

In [ ]:
for n in [10, 100, 10_000]:
    fracciones = [len(np.unique(rng.integers(0, n, n))) / n for _ in range(200)]
    print(f"Grupo de {n:>6}: en promedio aparece el {np.mean(fracciones):.1%} de los ejemplos")

<details><summary><b>👀 Ver respuesta</b></summary>

**b)** En promedio aparecen unos **6–7 de cada 10** (≈65 % con 10 nombres). Con grupos grandes la cifra se estabiliza en **≈63.2 %**: cada muestra bootstrap contiene unas **2/3 partes** de los ejemplos y deja fuera **≈1/3**.

Esos ejemplos que se quedan fuera se llaman **out-of-bag** (OOB, "fuera de la bolsa"). Guárdalos en la memoria: en la Sección 3 serán una validación gratis.
</details>

---
## 2. 📖 El concepto: bagging paso a paso

1. Haz **B** muestras bootstrap de los datos de entrenamiento (B = 100, 500…).
2. Entrena **un árbol profundo** con cada muestra (sin podar: mucha varianza, poco sesgo).
3. Para predecir, junta las predicciones: **voto** (clasificación) o **promedio** (regresión).

¿Por qué árboles **profundos**? Por lo que viste en la Lección 6.1: el promedio cancela la **varianza** pero no el **sesgo**. Un árbol profundo tiene poco sesgo y mucha varianza: justo lo que el promedio sabe arreglar.

Vamos a comprobarlo con las lunas ruidosas de la Lección 5.4, donde un árbol sin límites sobreajustaba.

In [ ]:
X_lunas, y_lunas = make_moons(n_samples=400, noise=0.35, random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X_lunas, y_lunas, test_size=0.3, random_state=1)

def dibujar_frontera(modelo, ax, titulo):
    xx, yy = np.meshgrid(np.linspace(-2, 3, 250), np.linspace(-1.7, 2.2, 250))
    zz = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap="coolwarm")
    ax.scatter(*X_tr.T, c=y_tr, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(titulo, fontsize=10); ax.set_xticks([]); ax.set_yticks([])

### 🔮 Ejercicio 2 — Un árbol contra cien
Compararemos **un árbol sin límites** contra **bagging de 100 árboles sin límites**.
1. ¿Cuál tendrá la frontera más "suave"?
2. En **entrenamiento**, ¿los dos tendrán ≈100 %?
3. En **prueba**, ¿cuál ganará y por cuántos puntos, más o menos?

✍️ *Mis predicciones:*

In [ ]:
arbol = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
bagging = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=0).fit(X_tr, y_tr)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (nombre, m) in zip(axes, [("1 árbol", arbol), ("Bagging de 100 árboles", bagging)]):
    dibujar_frontera(m, ax, f"{nombre}\ntrain {m.score(X_tr, y_tr):.0%} · test {m.score(X_te, y_te):.0%}")
    print(f"{nombre:<24} entrenamiento={m.score(X_tr, y_tr):.1%}  prueba={m.score(X_te, y_te):.1%}")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

1. El **bagging**: sus "islas" alrededor de puntos sueltos son más pequeñas y la frontera sigue mejor la forma de las lunas.
2. **Sí**, los dos ≈100 % en entrenamiento: cada árbol memoriza su muestra, y el conjunto también "recuerda" los datos.
3. En prueba gana el **bagging**: ≈84 % contra ≈77 %, unos 7 puntos. Mismo tipo de modelo, mismos datos: la única diferencia es **promediar**.

⚠️ Fíjate en que 100 % en entrenamiento **no** es aquí señal de alarma: el bosque generaliza bien igual. Para juzgar un ensamble de árboles, **mira siempre validación o prueba**, nunca el entrenamiento.
</details>

---
## 3. 📖 Validación gratis: el error *out-of-bag* (OOB)

Recuerda: cada árbol dejó fuera ~1/3 de los ejemplos. Para cada ejemplo, podemos preguntarle **solo a los árboles que no lo vieron** y comparar su voto con la respuesta real. Es como una validación cruzada que viene **incluida** con el entrenamiento, sin apartar datos ni entrenar de nuevo.

En scikit-learn basta con `oob_score=True`.

### ✋ Ejercicio 3 — ¿Se parece el OOB a la prueba?
Ejecuta la celda. ¿El puntaje OOB se parece más a la exactitud de **entrenamiento** (≈100 %) o a la de **prueba** (≈84 %)? ¿Por qué?

✍️ *Mi respuesta:*

In [ ]:
bagging_oob = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100,
                                oob_score=True, random_state=0).fit(X_tr, y_tr)
print(f"Entrenamiento : {bagging_oob.score(X_tr, y_tr):.1%}")
print(f"OOB           : {bagging_oob.oob_score_:.1%}   ← calculado SOLO con los datos de entrenamiento")
print(f"Prueba        : {bagging_oob.score(X_te, y_te):.1%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Se parece mucho más a la **prueba**: OOB ≈ **89 %** frente a ≈84 % en prueba (y 100 % en entrenamiento). Es honesto porque cada ejemplo lo juzgan solo árboles que **no** lo vieron. La diferencia de unos puntos es normal: con solo 120 ejemplos de prueba, unos pocos aciertos mueven el porcentaje.

Uso práctico: el OOB te da una estimación del desempeño **sin gastar datos** en validación. Muy útil cuando tienes pocos datos o quieres ajustar algo rápido. (Para decisiones finales, la validación cruzada sigue siendo el estándar, Lección 4.6.)
</details>

---
## 4. 📖 Random Forest: el truco de la pregunta prohibida

Bagging tiene un punto débil. Imagina que entre tus características hay **una muy fuerte** (en un banco, por ejemplo, "¿tiene deudas atrasadas?"). Aunque cada árbol vea datos distintos, **todos** empezarán preguntando por esa característica: los árboles se parecen mucho entre sí, sus errores se **correlacionan** y el voto rescata menos (el `COPIA` de la Lección 6.1).

**Random Forest** (Breiman, 2001) añade un segundo ingrediente de azar:

> En **cada división**, el árbol solo puede elegir entre un **subconjunto al azar** de las características (`max_features`).

A veces la característica estrella **no está disponible** y el árbol tiene que buscar otra pregunta. Los árboles quedan más **distintos** entre sí.

```
Bagging:        cada árbol → datos al azar     + todas las columnas en cada división
Random Forest:  cada árbol → datos al azar     + solo algunas columnas al azar en cada división
```

Para verlo, creamos datos donde se predice si un cliente compra con **una señal fuerte** y **11 pistas débiles**.

In [ ]:
rng_d = np.random.default_rng(0)
n = 2_000
X_pistas = rng_d.normal(size=(n, 12))                    # columna 0 = señal fuerte; 1–11 = pistas débiles
puntaje = 2 * X_pistas[:, 0] + 0.6 * X_pistas[:, 1:].sum(axis=1) + rng_d.normal(0, 1, n)
y_pistas = (puntaje > 0).astype(int)
nombres_pistas = ["senal_fuerte"] + [f"pista_{i}" for i in range(1, 12)]

Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_pistas, y_pistas, train_size=600, random_state=0)
print("Entrenamiento:", Xp_tr.shape, "· Prueba:", Xp_te.shape)

### 🔮 Ejercicio 4 — ¿Árboles peores, bosque mejor?
Entrenaremos tres bosques de 300 árboles que solo difieren en `max_features`:
- `None` → todas las columnas en cada división (= **bagging** puro)
- `"sqrt"` → √12 ≈ 3 columnas al azar (el valor por defecto en clasificación)
- `1` → **una** columna al azar (¡el árbol ni siquiera elige qué preguntar!)

Para cada bosque, la celda mide: qué % de árboles empieza preguntando por `senal_fuerte`, qué tan **parecidos** son los árboles entre sí (correlación media de sus predicciones), la exactitud de un **árbol individual** promedio y la del **bosque**.

Antes de ejecutar: al bajar `max_features`, ¿la exactitud de cada árbol **sube o baja**? ¿Y la del bosque?

✍️ *Mis predicciones:*

In [ ]:
filas = []
for mf in [None, "sqrt", 1]:     # ✏️ prueba también 6
    bosque = RandomForestClassifier(n_estimators=300, max_features=mf, random_state=0, n_jobs=-1).fit(Xp_tr, yp_tr)
    preds = np.array([arbol.predict(Xp_te) for arbol in bosque.estimators_])
    corr = np.corrcoef(preds)[np.triu_indices(len(preds), 1)].mean()
    filas.append({
        "max_features": str(mf),
        "% árboles que empiezan por senal_fuerte": np.mean([a.tree_.feature[0] == 0 for a in bosque.estimators_]),
        "parecido entre árboles (correlación)": corr,
        "exactitud de UN árbol (promedio)": np.mean([np.mean(p == yp_te) for p in preds]),
        "exactitud del BOSQUE": bosque.score(Xp_te, yp_te),
    })
pd.DataFrame(filas).set_index("max_features").round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

| `max_features` | empiezan por la señal | parecido | un árbol | bosque |
|---|---|---|---|---|
| None (bagging) | 100 % | 0.36 | 0.705 | 0.806 |
| sqrt | 25 % | 0.25 | 0.681 | 0.821 |
| 1 | 6 % | 0.12 | 0.628 | 0.835 |

¡La paradoja de Random Forest! Al bajar `max_features`, **cada árbol empeora** (ya no puede elegir siempre la mejor pregunta), pero los árboles se vuelven **más distintos** y el **bosque mejora**. Es la condición 2 de la Lección 6.1 en acción: **la diversidad puede valer más que la calidad individual**.

⚠️ Aquí `max_features=1` ganó, pero no es una regla: con muchas columnas inútiles, preguntar por una al azar desperdicia divisiones. Por eso `max_features` se **ajusta** (Sección 5).
</details>

---
## 5. 📖 Hiperparámetros: las perillas del bosque

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `n_estimators` | Número de árboles | Más estable y algo mejor, hasta estancarse; **más lento**. **No** sobreajusta | Resultados más ruidosos | 200–500 |
| `max_features` | Columnas candidatas en cada división | Árboles mejores pero más parecidos (→ bagging) | Árboles más distintos pero más débiles | `"sqrt"` (clasif.) · `1.0` o `0.3–0.5` (regresión) |
| `max_depth` | Profundidad máxima de cada árbol | Árboles más complejos | Árboles más simples (más sesgo) | `None` |
| `min_samples_leaf` | Ejemplos mínimos por hoja | Árboles más simples, menos ruido memorizado | Hojas de 1 ejemplo | 1; prueba 3–20 si los datos son ruidosos |
| `bootstrap` | ¿Muestras con reemplazo? | — | `False` = todos los árboles ven todos los datos (menos diversidad, sin OOB) | `True` |
| `max_samples` | Tamaño de cada muestra bootstrap | — | Árboles más distintos y entrenamiento más rápido | `None` (= 100 %) |
| `oob_score` | Calcular la validación OOB | — | — | `True` si quieres una estimación rápida |
| `class_weight` | Peso de cada clase | — | — | `None`, o `"balanced"` si hay desbalance |
| `n_jobs` | Núcleos del procesador a usar | Entrena más rápido | — | `-1` (todos) |

### 🧠 Regla mnemotécnica
> **Más árboles nunca hacen daño (solo cuestan tiempo). Las perillas que importan son las que cambian *cada* árbol: `max_features` y `min_samples_leaf`.**

### 🔮 Ejercicio 5 — ¿Sobreajusta un bosque con demasiados árboles?
En un árbol, más profundidad = más sobreajuste. ¿Pasará lo mismo con el **número de árboles**? Predice la forma de la curva de exactitud en prueba al pasar de 1 a 800 árboles:
- a) Sube y luego baja (forma de "U" invertida)
- b) Sube y luego se queda plana
- c) Baja todo el tiempo

✍️ *Mi apuesta:*

In [ ]:
N_ARBOLES = [1, 5, 10, 25, 50, 100, 200, 400, 800]
exactitudes = [RandomForestClassifier(n_estimators=n, random_state=0, n_jobs=-1).fit(Xp_tr, yp_tr).score(Xp_te, yp_te)
               for n in N_ARBOLES]

plt.semilogx(N_ARBOLES, exactitudes, "o-")
plt.xlabel("n_estimators (escala logarítmica)"); plt.ylabel("Exactitud en prueba"); plt.grid(alpha=.3)
plt.title("¿Más árboles = sobreajuste?"); plt.show()
print(dict(zip(N_ARBOLES, np.round(exactitudes, 3))))

<details><summary><b>👀 Ver respuesta</b></summary>

**b)** Sube rápido (de ≈0.70 con 1 árbol a ≈0.81 con 50) y luego se queda **plana** alrededor de 0.82–0.83, con pequeñas oscilaciones de azar. Agregar árboles es agregar votantes: el promedio se vuelve más estable, pero llega un punto en que ya no cambia.

Por eso `n_estimators` **no se "ajusta"** como las demás perillas: pon "suficientes" (unos cientos) y listo. Más allá de eso solo pagas en tiempo y memoria.
</details>

### 🔮 Ejercicio 6 — La profundidad: árbol solo contra bosque
En la Lección 5.4, subir `max_depth` de un árbol solo llevaba al sobreajuste. ¿Le pasará lo mismo a un **bosque**? Predice:
1. ¿Con qué profundidad tendrá su mejor resultado **el árbol solo**?
2. ¿Y el **bosque**: le dolerá dejar los árboles sin límite (`None`)?

✍️ *Mis predicciones:*

In [ ]:
PROFUNDIDADES = [1, 2, 4, 6, 8, 12, None]
for d in PROFUNDIDADES:
    solo = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xp_tr, yp_tr).score(Xp_te, yp_te)
    bosque = RandomForestClassifier(n_estimators=300, max_depth=d, random_state=0, n_jobs=-1).fit(Xp_tr, yp_tr).score(Xp_te, yp_te)
    print(f"max_depth={str(d):>4}   árbol solo={solo:.3f}   bosque={bosque:.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. El árbol solo alcanza su mejor valor con profundidad **6** (≈0.75) y después **empeora** (≈0.72 sin límite): sobreajuste clásico.
2. Al bosque **no le duele**: sube hasta ≈0.82 y se queda ahí incluso sin límite. Los árboles profundos memorizan ruido, pero cada uno memoriza un ruido **distinto** y el voto lo cancela.

Por eso el valor por defecto de `max_depth` en Random Forest es `None`. Y fíjate en la brecha: el bosque le gana al **mejor** árbol solo por unos 7 puntos.
</details>

### ✏️ Ejercicio 7 — `min_samples_leaf` y `max_features` con validación cruzada
Ajustemos las dos perillas importantes **sin mirar la prueba**, con validación cruzada sobre el entrenamiento (Lección 4.6). Cambia los valores de las listas y observa.

In [ ]:
VALORES_MSL = [1, 5, 20]           # ✏️ min_samples_leaf
VALORES_MF = [1, "sqrt", 0.5]      # ✏️ max_features (un número entero = columnas; un decimal = fracción)

tabla = pd.DataFrame(index=VALORES_MSL, columns=[str(v) for v in VALORES_MF], dtype=float)
for msl in VALORES_MSL:
    for mf in VALORES_MF:
        modelo = RandomForestClassifier(n_estimators=200, min_samples_leaf=msl, max_features=mf, random_state=0, n_jobs=-1)
        tabla.loc[msl, str(mf)] = cross_val_score(modelo, Xp_tr, yp_tr, cv=5).mean()
tabla.index.name = "min_samples_leaf ↓ / max_features →"
tabla.round(3)

✋ ¿Qué combinación elegirías? ¿Cuál de las dos perillas mueve más el resultado en estos datos?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Lo mejor es ≈**0.848** con `min_samples_leaf=5` y `max_features` = 1 o `"sqrt"` (empatan). Pero fíjate en lo **poco** que se mueve la tabla: casi todo está entre 0.83 y 0.85. Solo se cae (≈0.80) al combinar árboles muy simples (`min_samples_leaf=20`) con muchas columnas (`max_features=0.5`), es decir, árboles más simples **y** más parecidos.

Moraleja: **un Random Forest con valores razonables casi nunca sale mal**. Esa robustez es una de sus mayores virtudes. En datos **muy ruidosos**, en cambio, `min_samples_leaf` sí marca la diferencia (lo verás en la práctica integradora).
</details>

---
## 6. 📖 ¿Qué variables usa el bosque? Importancias con cuidado

Un bosque de 300 árboles no se puede leer como un solo árbol. Pero sí podemos preguntar **qué variables le importan**. Hay dos formas:

| Método | Cómo se calcula | Cuidado |
|---|---|---|
| **Por impureza** (`feature_importances_`) | Cuánto redujo la impureza cada variable, sumando en todas las divisiones de todos los árboles | Se calcula con los datos de **entrenamiento** y **favorece** a las variables con muchos valores distintos |
| **Por permutación** (`permutation_importance`) | Cuánto **empeora** el modelo si **barajas** una columna en datos de validación (rompiendo su relación con el objetivo) | Más lenta, pero honesta (la verás a fondo en la Lección 7.6) |

### 🔮 Ejercicio 8 — Dos columnas inútiles
Agregamos a los datos dos columnas **totalmente al azar**, sin ninguna relación con la compra:
- `es_fin_de_semana`: 0 o 1 al azar.
- `id_cliente`: un número distinto para cada cliente (2,000 valores distintos).

Según la importancia **por impureza**, ¿las dos tendrán importancia ≈0? ¿Alguna saldrá "importante"?

✍️ *Mi predicción:*

In [ ]:
df = pd.DataFrame(X_pistas, columns=nombres_pistas)
df["es_fin_de_semana"] = rng_d.integers(0, 2, n)       # ruido con 2 valores posibles
df["id_cliente"] = rng_d.permutation(n)                # ruido con 2,000 valores posibles
df_tr, df_te, ydf_tr, ydf_te = train_test_split(df, y_pistas, train_size=600, random_state=0)

bosque = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(df_tr, ydf_tr)
perm = permutation_importance(bosque, df_te, ydf_te, n_repeats=10, random_state=0)

importancias = pd.DataFrame({
    "por impureza": bosque.feature_importances_,
    "por permutación": perm.importances_mean,
}, index=df.columns).sort_values("por impureza", ascending=False)
importancias.round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

Por **impureza**, `id_cliente` recibe ≈**0.040**: casi lo mismo que una pista real (0.05–0.07) y **siete veces** más que `es_fin_de_semana` (≈0.006), siendo las dos igual de inútiles. Con 2,000 valores distintos, `id_cliente` ofrece muchísimos puntos de corte, y algunos separan "por casualidad" los datos de entrenamiento.

Por **permutación**, las dos columnas de ruido valen ≈**0** (barajarlas no empeora nada en datos nuevos), y `senal_fuerte` queda claramente en primer lugar.

Regla práctica: usa la importancia por impureza solo como **primera mirada** y confirma con **permutación** antes de sacar conclusiones o de contárselas a alguien.
</details>

### 📖 Un primo cercano: Extra Trees
`ExtraTreesClassifier` (*Extremely Randomized Trees*) lleva el azar un paso más allá: además de elegir columnas al azar, elige el **punto de corte al azar** en lugar de buscar el mejor. Árboles aún más distintos y más rápidos de entrenar. Se usa igual que `RandomForestClassifier` y a veces gana; vale la pena probarlo (lo harás en los retos de la práctica).

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| "Mi bosque tiene 100 % en entrenamiento: ¡sobreajuste!" | Es normal con árboles profundos | Juzga con OOB, validación cruzada o prueba |
| Bajar `n_estimators` "para evitar sobreajuste" | Mito: más árboles no sobreajustan | Deja 200–500 y ajusta `min_samples_leaf` / `max_features` |
| El bosque rinde igual que un árbol solo | Datos muy ruidosos: cada árbol memoriza demasiado ruido | Subir `min_samples_leaf` (5–20) |
| Una columna tipo ID o fecha sale "importante" | Sesgo de la importancia por impureza (o fuga, Lección 2.6) | Importancia por permutación; quitar identificadores |
| Predicciones planas fuera del rango visto (regresión) | Los árboles **no extrapolan** (Lección 5.4), y su promedio tampoco | Otro modelo si necesitas extrapolar |
| El modelo pesa cientos de MB y tarda en predecir | Muchos árboles muy profundos con muchos datos | Limitar `max_depth`/`min_samples_leaf`, menos árboles, o boosting (6.3–6.4) |

### 🔮 Ejercicio 9 — ¿Bosque sin azar?
Si pones `bootstrap=False` (todos los árboles ven **todos** los datos) **y** `max_features=None` (todas las columnas en cada división), ¿qué tan distintos serán los árboles entre sí? ¿Qué esperas de su exactitud comparada con un árbol solo?

✍️ *Mi predicción:*

In [ ]:
sin_azar = RandomForestClassifier(n_estimators=50, bootstrap=False, max_features=None, random_state=0).fit(Xp_tr, yp_tr)
preds = np.array([a.predict(Xp_te) for a in sin_azar.estimators_])
print(f"Coincidencia media de cada árbol con el primero: {np.mean(preds == preds[0]):.1%}")
print(f"Bosque 'sin azar': {sin_azar.score(Xp_te, yp_te):.3f}")
print(f"Un árbol solo    : {DecisionTreeClassifier(random_state=0).fit(Xp_tr, yp_tr).score(Xp_te, yp_te):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Los árboles salen **casi clones**: coinciden en ≈92 % de las predicciones (solo difieren cuando dos preguntas empatan y se desempata al azar). El "bosque" da ≈0.726, prácticamente lo mismo que un árbol solo (≈0.721), muy lejos del ≈0.82 de un Random Forest normal. Sin las **dos fuentes de azar**, no hay diversidad y no hay sabiduría de la multitud.
</details>

---
## 8. 📋 Ficha del modelo

| | Random Forest (y bagging) |
|---|---|
| **Tipo** | Supervisado: clasificación y regresión. Ensamble de tipo **bagging** |
| **Intuición** | Cientos de árboles, cada uno con datos y preguntas un poco distintos, que votan |
| **Cómo aprende** | Cada árbol se entrena (en paralelo) con una muestra bootstrap y, en cada división, solo con `max_features` columnas al azar |
| **Hiperparámetros clave** | `n_estimators` (suficientes), `max_features`, `min_samples_leaf` |
| **✅ Fortalezas** | Muy buen desempeño "de fábrica" · Difícil de arruinar · No necesita escalar · Validación OOB gratis · Paraleliza bien |
| **❌ Debilidades** | No se puede leer como un árbol · Pesado en memoria · No extrapola · Importancia por impureza sesgada · Suele quedar por debajo de un boosting bien ajustado |
| **Úsalo cuando…** | Quieres un modelo fuerte con poco ajuste, o una línea base seria para datos tabulares |
| **Evítalo cuando…** | Necesitas explicar cada decisión, el modelo debe ser muy liviano, o necesitas extrapolar |

## 9. 🧠 Resumen en 5 puntos
1. Una **muestra bootstrap** toma ejemplos con reemplazo: contiene ~63 % de los ejemplos y deja fuera ~37 % (**out-of-bag**).
2. **Bagging** entrena árboles profundos con muestras distintas y los combina: reduce la **varianza** sin aumentar el sesgo.
3. **Random Forest** añade azar en las columnas (`max_features`): árboles más débiles pero más **distintos** → bosque mejor.
4. **Más árboles no sobreajustan**; las perillas que importan son `max_features` y `min_samples_leaf`.
5. El error **OOB** es una validación gratis; la importancia **por impureza** engaña, confírmala con **permutación**.

---
## 10. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈35 min).

### 10.1 Repaso rápido (opción múltiple)

**1.** Tienes 1,000 ejemplos. ¿Cuántos ejemplos **distintos** esperas, aproximadamente, en una muestra bootstrap?
- a) 1,000 · b) ≈630 · c) ≈500 · d) ≈370

**2.** Tu Random Forest de 100 árboles tiene 100 % en entrenamiento y 88 % en OOB. ¿Qué es lo **menos** útil?
- a) Subir `min_samples_leaf` y comparar el OOB · b) Probar otros `max_features` · c) Bajar a 20 árboles para "evitar el sobreajuste"

**3.** ¿Qué diferencia a un Random Forest de bagging con árboles?
- a) Random Forest usa árboles podados · b) Random Forest elige columnas al azar en cada división · c) Random Forest entrena los árboles en secuencia · d) Ninguna

**4.** Bajas `max_features` de "todas" a 1. Lo más probable es que…
- a) Cada árbol mejore y el bosque también · b) Cada árbol empeore, pero los árboles se parezcan menos · c) Nada cambie

**5.** ¿Por qué el error OOB es una estimación honesta?
- a) Porque usa el conjunto de prueba · b) Porque cada ejemplo lo evalúan solo los árboles que no lo vieron al entrenar · c) Porque usa árboles podados

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** ≈63.2 % de los ejemplos aparece al menos una vez.
2. **c)** Más árboles no sobreajustan; quitar árboles solo hace el modelo más inestable. El 100 % en entrenamiento es normal en un bosque.
3. **b)** Ese es el "truco de la pregunta prohibida" (`max_features`).
4. **b)** Es la paradoja del Ejercicio 4. Si el bosque mejora o no depende de los datos: hay que medirlo.
5. **b)** Es como una validación cruzada incluida en el entrenamiento.
</details>

### 10.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: *"Random Forest usa árboles sin podar que sobreajustan; entonces el bosque también sobreajusta."* ¿Qué le responderías usando la idea de sesgo y varianza (Lección 4.5)?

**B. Caso de análisis 💼** Una aseguradora usa un Random Forest para estimar el costo de los siniestros de autos. Sus datos van de autos de $100,000 a $1,500,000. Mañana empezará a asegurar autos de lujo de **$4,000,000**. El equipo dice: *"No hay problema, el bosque es muy preciso (R² = 0.91)"*. ¿Qué riesgo ves? ¿Qué propondrías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Cada árbol sin podar tiene poco **sesgo** y mucha **varianza**. El promedio de muchos árboles **distintos** conserva el poco sesgo pero **cancela** gran parte de la varianza. El bosque no hereda el sobreajuste de cada árbol, porque cada uno memoriza un ruido diferente. (Prueba: Ejercicio 6, donde el bosque con `max_depth=None` no empeora.)

**B.** Los árboles **no extrapolan**: para un auto de $4,000,000 el bosque predecirá algo parecido a lo que vio para los autos más caros del entrenamiento (como la "mansión" de la Lección 5.4). Un R² de 0.91 se midió **dentro** del rango conocido y no dice nada fuera de él. Propuestas: no usar el modelo para ese segmento hasta tener datos; usar un modelo que sí capture la tendencia (p. ej. regresión lineal sobre el valor del auto) o predecir el costo **como porcentaje** del valor del auto; y monitorear esos casos aparte.
</details>

### 10.3 🧩 Práctica integradora: ¿le gana el bosque al árbol en la telefónica?

¿Recuerdas los 2,000 clientes de telefonía de la Lección 5.4? Allí un árbol con `max_depth` ≈ 5 llegó a ≈80 % en validación cruzada. Hoy veremos si un bosque lo supera. (Ya **sin** la columna `id_cliente` tramposa.)

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Los datos de siempre**

In [ ]:
rng_tel = np.random.default_rng(7)
n_tel = 2_000
tel = pd.DataFrame({
    "meses_cliente":    rng_tel.integers(1, 72, n_tel),
    "llamadas_soporte": rng_tel.poisson(1.5, n_tel),
    "cargo_mensual":    rng_tel.integers(200, 1_500, n_tel),
    "contrato_anual":   rng_tel.integers(0, 2, n_tel),
})
riesgo = (-2.0 + 1.1 * tel.llamadas_soporte - 0.07 * tel.meses_cliente
          + 0.0025 * tel.cargo_mensual - 2.2 * tel.contrato_anual)
tel["se_fue"] = (rng_tel.random(n_tel) < 1 / (1 + np.exp(-riesgo))).astype(int)

X_tel, y_tel = tel.drop(columns="se_fue"), tel["se_fue"]
X_tel_tr, X_tel_te, y_tel_tr, y_tel_te = train_test_split(X_tel, y_tel, test_size=0.25, stratify=y_tel, random_state=0)

cv_arbol = cross_val_score(DecisionTreeClassifier(max_depth=5, random_state=0), X_tel_tr, y_tel_tr, cv=5).mean()
print(f"Árbol max_depth=5 (validación cruzada): {cv_arbol:.3f}")

**Paso 2 🔮 — El bosque de fábrica.** Entrenamos un Random Forest con valores por defecto (300 árboles). Antes de ejecutar: ¿cuántos puntos le sacará al árbol?

✍️ *Mi apuesta:*

In [ ]:
bosque_tel = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=0, n_jobs=-1)
cv_bosque = cross_val_score(bosque_tel, X_tel_tr, y_tel_tr, cv=5).mean()
bosque_tel.fit(X_tel_tr, y_tel_tr)
print(f"Bosque por defecto → validación cruzada: {cv_bosque:.3f} · OOB: {bosque_tel.oob_score_:.3f} · entrenamiento: {bosque_tel.score(X_tel_tr, y_tel_tr):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

¡Sorpresa! El bosque de fábrica **no le gana** al árbol: ≈0.798 en validación cruzada (≈0.805 OOB) frente a ≈0.801 del árbol. Y tiene ≈100 % en entrenamiento. ¿Qué está pasando? Ve al paso 3.
</details>

**Paso 3 ✋ — Diagnóstico.** Mira la tabla de errores comunes (Sección 7). Estos datos tienen solo 4 columnas y **mucho ruido**: el comportamiento humano es poco predecible. ¿Qué fila describe el problema y qué perilla tocarías?

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver respuesta</b></summary>

La fila *"El bosque rinde igual que un árbol solo → datos muy ruidosos"*. Con hojas de 1 ejemplo, cada árbol memoriza muchísimo ruido, y con solo 4 columnas los árboles tampoco pueden ser muy distintos entre sí (poca diversidad). La perilla a tocar: **subir `min_samples_leaf`**, y probar `max_features` bajos para ganar diversidad.
</details>

**Paso 4 ✏️ — Ajusta con validación cruzada.** Completa las listas con los valores que quieras probar (ya hay unos de ejemplo).

In [ ]:
VALORES_MSL = [1, 5, 10, 20, 40]     # ✏️ min_samples_leaf
VALORES_MF = [1, 2, 4]               # ✏️ max_features (hay 4 columnas)

resultados = {}
for msl in VALORES_MSL:
    for mf in VALORES_MF:
        m = RandomForestClassifier(n_estimators=300, min_samples_leaf=msl, max_features=mf, random_state=0, n_jobs=-1)
        resultados[(msl, mf)] = cross_val_score(m, X_tel_tr, y_tel_tr, cv=5).mean()

tabla_tel = pd.Series(resultados).unstack()
tabla_tel.index.name, tabla_tel.columns.name = "min_samples_leaf", "max_features"
print("Mejor combinación:", max(resultados, key=resultados.get), f"→ {max(resultados.values()):.3f}")
tabla_tel.round(3)

<details><summary><b>👀 Ver respuesta</b></summary>

El mejor resultado es ≈**0.830** con `min_samples_leaf` = 5 o 10 y `max_features=1` (las dos empatan; elegimos `min_samples_leaf=5`, la primera). Son **3 puntos** más que el bosque de fábrica y que el árbol. Fíjate en el patrón:
- `min_samples_leaf=1` (de fábrica) es la **peor** fila: memoriza ruido.
- Con 40 vuelve a bajar: los árboles quedan demasiado simples (subajuste).
- `max_features=1` gana casi siempre: con pocas columnas, la diversidad vale oro.

Es la "U" del sesgo-varianza de siempre, ahora en el bosque.
</details>

**Paso 5 ▶️ — ¿Qué variables usa?** Calculamos la importancia por permutación del bosque ajustado, sobre el conjunto de prueba.

In [ ]:
MSL, MF = 5, 1     # ✏️ pon aquí tu mejor combinación
final = RandomForestClassifier(n_estimators=300, min_samples_leaf=MSL, max_features=MF, random_state=0, n_jobs=-1).fit(X_tel_tr, y_tel_tr)
perm_tel = permutation_importance(final, X_tel_te, y_tel_te, n_repeats=10, random_state=0)
pd.Series(perm_tel.importances_mean, index=X_tel.columns).sort_values(ascending=False).round(3)

✋ ¿Cuál es la variable más importante? ¿Coincide con lo que te mostró el árbol de 3 niveles de la Lección 5.4?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Empatan arriba **`llamadas_soporte`** y **`meses_cliente`** (≈0.08 cada una), seguidas de `contrato_anual` (≈0.05) y `cargo_mensual` (≈0.04). Coincide con las reglas del árbol de la Lección 5.4, que preguntaba sobre todo por llamadas a soporte, antigüedad y contrato anual, y solo al final por el cargo mensual. El bosque **no te da reglas** que leer, pero sí te dice **en qué fijarte**.
</details>

**Paso 6 ▶️ — Examen final, una sola vez.**

In [ ]:
arbol_final = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X_tel_tr, y_tel_tr)
print(f"Árbol (max_depth=5)            en prueba: {arbol_final.score(X_tel_te, y_tel_te):.3f}")
print(f"Bosque por defecto             en prueba: {bosque_tel.score(X_tel_te, y_tel_te):.3f}")
print(f"Bosque ajustado (msl={MSL}, mf={MF}) en prueba: {final.score(X_tel_te, y_tel_te):.3f}")
print(f"Modelo 'todos se quedan'       en prueba: {1 - y_tel_te.mean():.3f}")

✋ ¿Se confirma en prueba lo que dijo la validación cruzada? Escribe una frase para el gerente de la telefónica.

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Sí: el bosque ajustado queda primero (≈**0.828**), por delante del bosque de fábrica (≈0.814) y del árbol (≈0.810), y todos le ganan a "todos se quedan" (≈0.732). La ventaja en prueba (≈2 puntos) es algo menor que en validación cruzada, lo normal con solo 500 clientes de prueba. Frase posible: *"Un bosque de 300 árboles bien ajustado acierta en ~83 de cada 100 clientes, unos 2 puntos más que un solo árbol; a cambio, ya no podemos leer sus reglas, pero sabemos que lo que más pesa son las llamadas a soporte y la antigüedad."*

🚀 **Retos opcionales:** (1) Prueba `ExtraTreesClassifier` con los mismos valores. (2) Agrega `class_weight="balanced"` y mira el *recall* de los que se van (Lección 4.2).
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, "Random Forests Part 1 & 2" (inglés, con subtítulos).
- 📄 **Artículo original:** Leo Breiman (2001), *Random Forests*, Machine Learning 45: muy legible para ser un artículo científico.
- 📘 **Documentación:** scikit-learn, guía de usuario, secciones *Bagging meta-estimator* y *Random forests*; y el ejemplo *Permutation Importance vs Random Forest Feature Importance (MDI)*.

---
⬅️ **Anterior:** [6.1 Por qué combinar modelos funciona: la sabiduría de la multitud](6.1-por-que-combinar-modelos.ipynb) · ➡️ **Siguiente:** [6.3 Boosting: aprender de los errores (AdaBoost y Gradient Boosting)](6.3-boosting-adaboost-y-gradient-boosting.ipynb)